# 聪明钱指数因子 (Smart Money Index - SMI) v1

## 因子逻辑

### 经济学直觉
- SMI 传统概念: 比较开盘时段（散户情绪驱动）与收盘时段（机构/聪明钱驱动）的市场表现
- 本因子简化实现: 当日内收益率为正且放量时，标记为聪明钱买入 (+1)；当日内收益率为负且放量时，标记为聪明钱卖出 (-1)
- 核心思想: 重大价格变动若伴随异常放量，更可能是知情交易者（聪明钱）的行为
- 滚动加总后，正值表示聪明钱净买入，负值表示聪明钱净卖出

### 计算公式
$$\text{SMI}_t = \begin{cases}
+1 & \text{if } \text{intraday_return}_t > 0 \text{ and } \text{volume}_t > \text{MA5(volume)}_t \\
-1 & \text{if } \text{intraday_return}_t < 0 \text{ and } \text{volume}_t > \text{MA5(volume)}_t \\
0 & \text{otherwise}
\end{cases}$$

$$\text{SMI_Score} = \sum_{i=1}^{20} \text{SMI}_{t-i+1}$$

### 因子方向
因子 = +SMI_Score (聪明钱净买入 → 看涨信号 → 预期正向收益)

### 参数
- 成交量 MA: 5 个交易日
- 滚动加总: 20 个交易日
- 缓冲: 向前多取 70 自然日

In [ ]:
def main(datasources, start_date, end_date):
    """
    聪明钱指数因子 (SMI) v1
    基于日内收益率方向与放量条件的聪明钱行为判别。
    因子方向: 值越大 = 预期收益越高 (聪明钱净买入 = 看涨信号)。
    """
    import time
    import numpy as np
    import pandas as pd
    import dai
    import structlog

    logger = structlog.get_logger()
    t_total = time.time()

    bar1m_table = datasources['bar1m']
    N_SMI = 20   # 滚动加总窗口
    N_VOL = 5    # 成交量 MA 窗口

    BUFFER_DAYS = N_SMI * 3 + N_VOL * 3 + 10
    query_start = pd.to_datetime(start_date) - pd.Timedelta(days=BUFFER_DAYS)

    # ============================================================
    # Step 1: SQL 日频聚合 OHLCV
    # ============================================================
    logger.info("Step 1: SQL 日频聚合 OHLCV")
    t1 = time.time()

    sql = f"""
    SELECT
        date::DATE::DATETIME AS trading_day,
        instrument,
        first(open ORDER BY date) AS open_p,
        last(close ORDER BY date) AS close_p,
        MAX(high) AS high_p,
        MIN(low) AS low_p,
        SUM(volume) AS total_volume,
        SUM(amount) AS total_amount
    FROM {bar1m_table}
    GROUP BY date::DATE::DATETIME, instrument
    ORDER BY trading_day, instrument
    """

    daily = dai.query(
        sql,
        filters={'date': [query_start.strftime('%Y-%m-%d %H:%M:%S'), end_date]},
        compression=True
    ).df()

    daily['trading_day'] = pd.to_datetime(daily['trading_day'])
    daily['instrument'] = daily['instrument'].astype(str)

    for col in ['open_p', 'close_p', 'high_p', 'low_p', 'total_volume', 'total_amount']:
        daily[col] = pd.to_numeric(daily[col], errors='coerce')

    logger.info(f"  -> 日频聚合: {len(daily)} 行, {daily['instrument'].nunique()} 标的, "
                f"耗时 {time.time()-t1:.1f}s")

    # ============================================================
    # Step 2: 计算 SMI 信号
    # ============================================================
    logger.info("Step 2: 计算 SMI 信号")
    t2 = time.time()

    daily = daily.sort_values(['instrument', 'trading_day']).reset_index(drop=True)
    eps = 1e-8

    # 日内收益率 (close - open) / open
    daily['intraday_ret'] = (daily['close_p'] - daily['open_p']) / (daily['open_p'] + eps)

    # 成交量 5 日均线
    daily['vol_ma5'] = daily.groupby('instrument')['total_volume'].transform(
        lambda x: x.rolling(N_VOL, min_periods=N_VOL).mean()
    )

    # SMI 信号: 条件判断
    smi_up = (daily['intraday_ret'] > eps) & (daily['total_volume'] > daily['vol_ma5'])
    smi_down = (daily['intraday_ret'] < -eps) & (daily['total_volume'] > daily['vol_ma5'])

    daily['smi_signal'] = 0
    daily.loc[smi_up, 'smi_signal'] = 1
    daily.loc[smi_down, 'smi_signal'] = -1

    # 滚动 20 日加总
    daily['smi_score'] = daily.groupby('instrument')['smi_signal'].transform(
        lambda x: x.rolling(N_SMI, min_periods=max(5, N_SMI//2)).sum()
    )

    # 因子方向: +smi_score
    daily['factor_raw'] = -daily['smi_score']

    valid = daily['smi_score'].notna().sum()
    logger.info(f"  -> SMI 计算完成: {valid}/{len(daily)} 有效, 耗时 {time.time()-t2:.1f}s")

    # ============================================================
    # Step 3: 极端值处理 + 截面标准化
    # ============================================================
    logger.info("Step 3: 极端值处理 + 截面标准化")
    t3 = time.time()

    daily['factor_raw'] = daily['factor_raw'].replace([np.inf, -np.inf], np.nan)

    q1, q99 = daily['factor_raw'].quantile(0.01), daily['factor_raw'].quantile(0.99)
    daily['factor_raw'] = daily['factor_raw'].clip(q1, q99)

    daily['factor'] = daily.groupby('trading_day')['factor_raw'].transform(
        lambda x: (x - x.mean()) / (x.std() + eps)
    )

    logger.info(f"  -> 标准化完成, 耗时 {time.time()-t3:.1f}s")

    # ============================================================
    # Step 4: 裁回评估区间 + 对齐成分股 + 输出
    # ============================================================
    logger.info("Step 4: 裁回区间 + 对齐成分股输出")
    t4 = time.time()

    daily = daily[
        (daily['trading_day'] >= pd.to_datetime(start_date)) &
        (daily['trading_day'] <= pd.to_datetime(end_date))
    ]

    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()
    stk_pool['instrument'] = stk_pool['instrument'].astype(str)
    stk_pool['date'] = pd.to_datetime(stk_pool['date'])

    result = daily.rename(columns={'trading_day': 'date'})
    result = pd.merge(
        result[['date', 'instrument', 'factor']],
        stk_pool,
        how='inner', on=['date', 'instrument']
    )

    result['factor'] = result['factor'].replace([np.inf, -np.inf], np.nan)
    result = result.dropna(subset=['factor']).reset_index(drop=True)[['date', 'instrument', 'factor']]

    logger.info(f"  -> 对齐后: {len(result)} 行, {result['date'].nunique()} 交易日, "
                f"{result['instrument'].nunique()} 标的, 耗时 {time.time()-t4:.1f}s")
    logger.info(f"因子构建完成! 总耗时 {time.time()-t_total:.1f}s")
    return result


if __name__ == '__main__':
    from bigmodule import M
    import dai
    import structlog

    logger = structlog.get_logger()

    datasources = {
        'bar1m': 'bigalpha_2026_stock_bar1m_selftest',
        'financial': 'bigalpha_2026_financial'
    }
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-10-31 23:59:59'

    logger.info(f"计算因子，测试区间：{start_date} ~ {end_date}")
    factor_data = main(datasources, start_date, end_date)

    logger.info(f"读取因子库，区间：{start_date} ~ {end_date}")
    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={'date': [start_date, end_date]},
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )